# Imbalanced Data

When one class is rare, rebalance the training data so a model learns to catch it.

This notebook comes from the [Imbalanced Data](https://pabl0aranda.github.io/Atlas-ML/data-preparation/imbalanced-data/) page on Atlas ML, an interactive guide to machine learning. Run the cells from top to bottom.

In [ ]:
%pip install --quiet imbalanced-learn

## Example data

The code on the page assumes some data already exists. This cell creates it.

In [ ]:
import numpy as np

# Two overlapping clouds of 1,000 samples, one in ten from the rare class, like the sandbox.
rng = np.random.default_rng(7)
y = (np.arange(1000) % 10 == 0).astype(int)
centers = np.where(y[:, None] == 1, 0.45, -0.15)
X = centers + rng.normal(0, 0.24, size=(1000, 2))
print("Rare samples:", y.sum(), "of", len(y))

## Code from the page

In [ ]:
from imblearn.over_sampling import SMOTE
from imblearn.pipeline import make_pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, stratify=y, random_state=0)

# The pipeline applies SMOTE while fitting only, never to data it predicts.
smote_model = make_pipeline(SMOTE(k_neighbors=5, random_state=0), LogisticRegression())
smote_model.fit(X_train, y_train)

# Class weights need no resampling at all.
weighted_model = LogisticRegression(class_weight="balanced").fit(X_train, y_train)

for model in (smote_model, weighted_model):
    print(classification_report(y_test, model.predict(X_test), digits=3))

## Try it

In [ ]:
# The same model without rebalancing, for comparison. Look at recall for class 1.
plain_model = LogisticRegression().fit(X_train, y_train)
print(classification_report(y_test, plain_model.predict(X_test), digits=3))